# LangChain Chains

Chain은 한 작업의 출력을 다음 작업의 입력으로 넘기도록 `Runnable`을 연결한 실행 경로이다. 프롬프트, 채팅 모델, 출력 파서, 분기와 대화 이력처럼 역할이 다른 구성 요소를 같은 실행 방식으로 연결할 수 있다.


Chain은 단계마다 입력과 출력 형식을 명확히 하여 재사용과 점검을 쉽게 만든다. 단순 질문 응답, 번역 뒤 요약, 입력별 처리 분기, 사용자별 대화 문맥처럼 이전 결과가 다음 단계에 영향을 주는 LLM 애플리케이션에 사용한다.

실습은 문자열·딕셔너리·`AIMessage`가 체인 사이에서 어떻게 바뀌는지 확인한다. 모델 문장은 생성 결과이므로 과목 지식이나 중요한 의사결정의 정답으로 단정하지 않고 별도 근거로 검증해야 한다.


## LCEL과 Runnable의 입력·출력 규칙

`Runnable`은 입력을 받아 작업을 실행하고 출력을 반환하는 공통 실행 단위이다. 프롬프트, 모델, 출력 파서와 Python 함수는 역할이 다르지만 모두 Runnable 방식으로 실행하고 연결할 수 있다.

여기서 **입력·출력 규칙**은 각 Runnable이 어떤 값을 받고 어떤 값을 반환하는지를 의미한다. 앞 단계의 출력 형식이 다음 단계의 입력 형식과 맞아야 두 작업을 자연스럽게 연결할 수 있다.

### Runnable의 실행 방법

- `invoke(input)`: 입력 하나를 실행하고 출력 하나를 반환한다.
- `batch([input1, input2])`: 여러 입력을 실행하고 같은 순서의 출력 목록을 반환한다.
- `stream(input)`: 생성되는 출력 조각을 준비되는 순서대로 반환한다.


### Chain과 LCEL

- `Chain`: Runnable을 두 개 이상 연결한 전체 처리 경로이다. 특정 Chain 클래스 하나만 의미하지 않는다.
- `LCEL`: LangChain Expression Language의 약자이다. `|` 연산자로 Runnable의 실행 순서를 표현한다.
- `|`: 왼쪽 Runnable의 출력을 오른쪽 Runnable의 입력으로 전달한다.

다음 체인은 입력을 세 번 변환하여 최종 문자열을 만든다.

`prompt | model | output_parser`

1. `PromptTemplate`은 딕셔너리나 문자열을 완성된 프롬프트 객체인 `PromptValue`로 바꾼다.
2. `ChatOpenAI`는 프롬프트를 받아 `AIMessage`를 반환한다.
3. `StrOutputParser`는 `AIMessage`에서 텍스트만 꺼내 `str`로 반환한다.

자료형의 변화는 다음과 같다.

`dict 또는 str → PromptValue → AIMessage → str`

`PromptValue`는 변수 치환이 끝난 프롬프트를 담고 있으며, 다음 `ChatOpenAI`가 읽을 메시지 형태로 변환할 수 있다.

### 출력 형식을 확인하는 이유

- 다음 단계가 문자열을 받아야 하면 `StrOutputParser`로 `AIMessage`를 `str`로 바꾼다.
- 모델 응답을 대화 이력에 저장해야 하면 `AIMessage`가 필요하므로 출력 파서를 연결하지 않는다.
- 분기마다 반환 형식이 다르면 뒤의 코드가 같은 방식으로 결과를 처리하기 어렵다.

Runnable의 공통 인터페이스와 조합 가능한 구성 요소는 [LangChain Runnables 공식 문서](https://reference.langchain.com/python/langchain-core/runnables)에서 확인할 수 있다.

### 체인 선택 기준

- **Simple Chain**: 하나의 프롬프트와 모델 호출로 문자열 응답이 필요할 때 사용한다.
- **Sequential Chain**: 앞 작업의 문자열 결과가 다음 작업의 입력이어야 할 때 사용한다.
- **Conditional Chain**: 입력 특징에 따라 처리 경로가 달라질 때 사용한다.
- **RunnableBranch**: 조건을 순서대로 검사해 처음 `True`인 체인을 선택할 때 사용한다.
- **RunnablePassthrough**: 원본 입력을 보존하면서 다음 단계가 요구하는 딕셔너리 구조로 포장할 때 사용한다.
- **RunnableWithMessageHistory**: 기존 LCEL 체인에 세션 이력을 연결하는 호환용 래퍼이다. 현재 버전에서는 신규 코드에 권장되지 않으므로 동작 원리를 확인한 뒤 신규 구현은 LangGraph persistence(상태 저장·복원)를 사용한다.


### LangChain 패키지 설치

체인 예제에 필요한 `langchain`, OpenAI 연동용 `langchain-openai`, `.env` 로딩용 `python-dotenv`를 설치한다. `-U`는 이미 설치된 패키지를 현재 호환 버전으로 갱신하며, 설치 로그의 버전은 실행 시점에 따라 달라질 수 있다.


In [ ]:
# %pip install -U langchain langchain-openai python-dotenv


### `.env`에서 OpenAI 인증과 모델 이름 읽기

`.env`는 API 키와 모델 설정을 코드 밖에서 관리하는 파일이다. 아래 셀은 `OPENAI_API_KEY`의 존재만 확인하고, `OPENAI_CHAT_MODEL`을 이후 `ChatOpenAI` 객체가 사용할 모델 ID로 저장한다. 키의 실제 값은 출력하지 않는다.


In [1]:
import os

from debugpy.launcher import output
from dotenv import find_dotenv, load_dotenv

dotenv_path = find_dotenv(usecwd=True)
if dotenv_path:
    # override=False는 운영체제에 이미 등록된 값을 .env 값으로 덮어쓰지 않는다.
    load_dotenv(dotenv_path, override=False)

if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('OPENAI_API_KEY가 없거나 비어 있다. .env 또는 운영체제 환경 변수에 설정한다.')

CHAT_MODEL_NAME = os.getenv('OPENAI_CHAT_MODEL', 'gpt-5.6-luna')


## Simple Chain

Simple Chain은 하나의 입력을 하나의 프롬프트와 모델에 전달해 최종 문자열을 얻는 가장 짧은 체인이다.

입력 `{'country': '대한민국'}`은 PromptTemplate에 들어가고, ChatOpenAI의 AIMessage는 StrOutputParser를 거쳐 문자열이 된다.

단일 질문 응답처럼 다음 단계가 텍스트만 필요할 때 이 구조를 선택한다. 다음 단계가 대화 메시지 전체나 여러 필드를 필요로 하면 파서를 생략하거나 딕셔너리 구조를 유지해야 한다.


### 국가 이름을 수도 질문으로 바꾸고 문자열 응답 받기

`PromptTemplate`은 `template` 문자열의 자리표시자에 입력값을 넣어 모델용 프롬프트를 만드는 Runnable이다.
여기서 `input_variables=['country']`는 `invoke()`가 받을 딕셔너리 key를 정하고, 완성된 프롬프트는 다음 `ChatOpenAI`의 입력이 된다.

`ChatOpenAI`는 프롬프트 메시지를 OpenAI 모델에 전달해 `AIMessage`를 반환하는 채팅 모델 Runnable이다.
`StrOutputParser`는 그 객체에서 텍스트만 꺼내 문자열로 바꾸므로, 다음 Sequential Chain이 중간 결과를 문자열 입력으로 바로 사용할 수 있다. 출력은 모델 생성 결과이므로 사실 여부는 신뢰할 수 있는 자료로 따로 확인해야 한다.


In [5]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 전달 받은 값을 이용해서 template 형태로 변경
# 이때, 데이터 타입은 PromptValue로 변함
prompt = PromptTemplate(
    template="{country}의 수도는 어디인가?",
    input_variables=["country"]
)

# print( prompt.invoke({"country":"대한민국"}) )

# LLM 요청 후 응답 데이터의 타입은 AIMessage
llm = ChatOpenAI(
    model_name=CHAT_MODEL_NAME,
    use_responses_api=True
)

# AIMessage에서 응답 메시지(str)만 꺼내서 반환
output_parser = StrOutputParser()

chain = prompt | llm | output_parser

# dict 입력 -> PromptValue -> AIMessage -> str 출력
response = chain.invoke({"country":"뉴질랜드"})
print(response)

뉴질랜드의 수도는 **웰링턴(Wellington)**입니다.


## Sequential Chain

Sequential Chain은 앞 체인의 출력이 뒤 체인의 입력이 되는 직렬 처리이다.
여기서는 1) 영어 문장을 번역해 문자열로 얻고, 2) 그 문자열을 요약 체인에 전달한다.
두 체인 사이에 StrOutputParser를 두면 번역 모델의 AIMessage가 요약 프롬프트가 받을 수 있는 문자열로 바뀐다.

번역, 분류, 요약처럼 작업 순서가 고정되고 앞 결과 없이는 다음 작업을 할 수 없을 때 선택한다. 독립 작업을 동시에 실행하거나 여러 키를 함께 전달해야 하면 다른 Runnable 조합을 선택한다.


### 번역 결과를 요약 입력으로 연결하기

`sentence` 문자열이 translation_chain에 들어가면 한국어 번역 문자열이 나오고, 그 문자열이 summary_chain의 `text` 입력이 된다. 아래에서는 중간 결과를 먼저 관찰해 앞 단계의 출력 형식이 다음 단계의 입력과 맞는지 확인한다.


In [6]:
sentence = """
One limitation of LLMs is their lack of contextual information (e.g., access to some specific documents or emails). You can combat this by giving LLMs access to the specific external data.
For this, you first need to load the external data with a document loader. LangChain provides a variety of loaders for different types of documents ranging from PDFs and emails to websites and YouTube videos.
"""

translation_prompt = PromptTemplate(
    template='다음 문장을 한국어로 번역하라: \n\n{text}',
    input_variables=['text']
)

llm = ChatOpenAI(
    model_name=CHAT_MODEL_NAME,
    use_responses_api=True
)

output_parser = StrOutputParser()

# 번역 체인
translation_chain = translation_prompt | llm | output_parser

translated_text = translation_chain.invoke({"text": sentence})
print(translated_text)

LLM의 한 가지 한계는 맥락 정보가 부족하다는 점입니다(예: 특정 문서나 이메일에 접근할 수 없음). 이러한 문제는 LLM이 특정 외부 데이터에 접근할 수 있도록 함으로써 해결할 수 있습니다.

이를 위해 먼저 문서 로더를 사용해 외부 데이터를 불러와야 합니다. LangChain은 PDF와 이메일부터 웹사이트 및 YouTube 동영상에 이르기까지 다양한 유형의 문서를 지원하는 여러 로더를 제공합니다.


In [7]:
# 번역된 내용을 전달받아 요약하는 체인
summary_prompt = PromptTemplate(
    template="다음 문장을 한 문장으로 짧게 요약하라:\n\n{text}",
    input_variables=["text"]
)

summary_chain = summary_prompt | llm | output_parser

summary_text = summary_chain.invoke(translated_text)
print(summary_text)

LLM의 맥락 부족 문제는 LangChain의 다양한 문서 로더로 외부 데이터를 불러와 해결할 수 있습니다.


### RunnableSequence로 번역과 요약을 하나의 체인으로 실행하기

앞 셀의 `translation_chain`과 `summary_chain`은 모두 문자열을 받고 문자열을 반환한다. `RunnableSequence`는 두 Runnable을 같은 순서로 묶어 원문 문자열 하나만으로 번역 뒤 요약까지 실행한다.


In [8]:
from langchain_core.runnables import RunnableSequence

chain = RunnableSequence(translation_chain, summary_chain)

# sentence -> 번역 체인 -> 번역 결과 -> 요약 체인 -> 요약 결과 반환
result = chain.invoke(sentence)
print(result)

LLM의 맥락 부족 문제는 LangChain의 다양한 문서 로더로 외부 데이터를 불러와 보완할 수 있습니다.


## Conditional Chain과 RunnableBranch

Sequential Chain은 작업 순서가 고정될 때 적합하지만, 사용자의 요청 형식에 따라 다른 프롬프트가 필요하면 한 줄의 경로로 처리하기 어렵다.

Conditional Chain은 이 한계를 해결하기 위해 입력 특징에 따라 서로 다른 체인을 실행하는 구조이다.

`RunnableBranch`는 조건 함수와 체인 쌍을 위에서 아래 순서로 검사하고, **처음 `True`를 반환한 분기**를 실행한다. 어느 조건도 참이 아니면 기본 체인을 실행한다.

입력 형식은 `{'text': '...'}` 딕셔너리로 통일한다. 조건 함수는 `text`를 꺼내 불리언을 반환하고, 선택된 프롬프트 체인은 문자열을 반환한다. 입력 주제·형식에 따라 처리 전략이 달라질 때 선택하며, 실제 의도 분류에는 단순 문자열 규칙의 오분류 가능성을 보완해야 한다.


### 채점 요청과 일반 질문을 서로 다른 체인으로 처리하기

채점 체인과 기본 답변 체인은 같은 `text` 딕셔너리를 받지만 프롬프트가 다르다. 두 체인 모두 마지막에 StrOutputParser를 두어 RunnableBranch의 최종 반환값을 문자열로 통일한다.


In [9]:
llm = ChatOpenAI(model=CHAT_MODEL_NAME, use_responses_api=True)
output_parser = StrOutputParser()

grading_prompt = PromptTemplate(
    template='당신은 친절한 채점자이다. 아래 답변을 1~5점으로 평가하라:\n\n{text}',
    input_variables=['text']
)

grading_chain = grading_prompt | llm | output_parser
grading_result = grading_chain.invoke(
    {'text': 'AI는 인공지능을 의미하며, 비전 처리와 자연어 생성을 할 수 있다.'}
)
print(grading_result)


# 기본 프롬프트
default_prompt = PromptTemplate(
    template='당신은 사용자의 질문에 답변하는 친절한 챗봇이다:\n\n{text}',
    input_variables=['text']
)
default_chain = default_prompt | llm | output_parser
default_result = default_chain.invoke({'text': '인공지능은 무엇인가?'})
print(default_result)

**평가: 4점/5점**

AI가 인공지능을 의미한다는 설명과 대표적인 기능인 **비전 처리** 및 **자연어 생성**을 잘 언급했습니다. 다만 AI는 이외에도 학습, 추론, 음성 인식, 의사결정 등 다양한 기능을 포함하므로 설명이 조금 더 포괄적이면 더 좋겠습니다.
인공지능(AI)은 컴퓨터가 사람처럼 **학습하고, 판단하고, 문제를 해결하며, 언어·이미지·음성 등을 이해하도록 만드는 기술**입니다.

예를 들어 인공지능은 다음과 같은 일을 할 수 있습니다.

- 질문에 답하기
- 사진 속 사물 인식하기
- 번역하기
- 음성 명령 이해하기
- 추천 상품이나 영상 제시하기
- 복잡한 데이터를 분석하기
- 글·그림·음악 만들기

인공지능은 많은 데이터를 학습해 일정한 패턴을 찾고, 이를 바탕으로 결과를 예측하거나 새로운 내용을 생성합니다. 다만 사람처럼 의식이나 감정을 가진 존재라기보다는, 학습한 정보와 프로그램에 따라 작동하는 기술입니다.


In [11]:
# text가 "채점" 이라는 단어로 시작하는지 검사하는 함수
def grading_routing_fn(input_dict) -> bool:

    text:str = input_dict.get('text', '')

    return text.strip().startswith("채점") # T/F


### `text` 규칙을 불리언으로 바꿔 첫 번째 분기 선택하기

`grading_routing_fn`은 `input_dict['text']`를 검사해 `채점`으로 시작하면 `True`를 반환한다.

RunnableBranch는 이 불리언을 보고 처음 참인 grading_chain을 선택하고, 거짓이면 default_chain으로 넘어간다.


In [12]:
from langchain_core.runnables import RunnableBranch

# grading_routing_fn 함수 반환 값이 True이면 grading_chain
# 아니면 default_chain 수행
cond_chain = RunnableBranch(
    (grading_routing_fn, grading_chain),
    default_chain
)

cond_result = cond_chain.invoke({
    "text":'채점: LangChain은 LLM과 외부 데이터, 다양한 도구를 연결하여 AI 애플리케이션을 쉽게 개발할 수 있게하는 프레임워크다.'
})

print(cond_result)

cond_result = cond_chain.invoke({
    "text":'SQLD 합격 기준'
})

print(cond_result)

**점수: 4/5점**

LangChain을 **LLM과 외부 데이터 및 도구를 연결해 AI 애플리케이션 개발을 돕는 프레임워크**라고 설명한 점은 정확합니다. 다만 프롬프트 관리, 체인 구성, 에이전트, 검색·검색증강생성(RAG) 등을 지원한다는 구체적인 기능이 추가되면 더 완전한 답변이 될 수 있습니다.
SQLD 합격 기준은 다음과 같습니다.

- **총점 60점 이상**
- **과목별 40% 이상 득점**(과락 기준)

출제 구성은 보통 다음과 같습니다.

| 과목 | 문항 수 | 배점 | 과락 기준 |
|---|---:|---:|---:|
| 1과목: 데이터 모델링의 이해 | 10문항 | 20점 | 8점 미만 |
| 2과목: SQL 기본 및 활용 | 40문항 | 80점 | 32점 미만 |
| **합계** | **50문항** | **100점** | **총점 60점 이상** |

문항당 2점이므로, 과목별로는 **1과목 4문항 이상**, **2과목 16문항 이상** 맞혀야 하며, 동시에 전체 **30문항 이상** 정답이면 합격 기준을 충족합니다.


### 숫자가 있는 입력을 수학 체인으로 분기하기

이번에는 숫자가 하나라도 있는지 검사하는 두 번째 조건을 추가한다. RunnableBranch는 채점 조건을 먼저 검사하고 수학 조건을 검사하므로 두 조건이 동시에 참이면 앞의 채점 체인이 선택된다.

입력 문자열의 `^`는 LLM에게 거듭제곱을 설명하기 위한 수학 표기이다. 이 셀에서는 `^`를 Python 연산자로 계산하지 않는다. Python 표현식 `3 ^ 3`에서 `^`는 거듭제곱이 아니라 비트 XOR 연산자이므로 두 의미를 혼동하면 안 된다.


In [14]:

# from_template(): {text} 자리에서 input_variables=['text'] 자동 추론
math_prompt = PromptTemplate.from_template(
    template='당신은 제공된 수식을 단계별로 풀어 최종 답안을 출력한다.\n\n{text}',
)

math_chain = math_prompt | llm | output_parser

def math_routing_fn(input_dict) -> bool:
    text = input_dict.get('text','')

    # text를 순차적으로 한 글자씩 뽑아내어(char) 숫자가 맞는지 확인
    # 숫자가 하나라도 있으면(any) True 반환
    return any(char.isdigit() for char in text)

cond_chain = RunnableBranch(
    (grading_routing_fn, cond_chain),
    (math_routing_fn, math_chain),
    default_chain
)

math_result = cond_chain.invoke({'text': '2^16은 얼마인가?'})
print(math_result)

\(2^{16}\)을 계산하면:

\[
2^{16}=2^8 \times 2^8=256 \times 256=65{,}536
\]

따라서 최종 답은 **65,536**입니다.


## RunnablePassthrough

`RunnablePassthrough`는 입력값을 **수정하지 않고 그대로 출력하는 Runnable**이다.

```text
입력값 → RunnablePassthrough → 같은 값
```

### 사용하는 이유

- 원본 입력을 뒤 단계까지 유지할 때 사용한다.
- 같은 입력으로 원본 값과 변환된 값을 함께 만들 때 사용한다.
- 다음 Runnable이 딕셔너리를 요구할 때 특정 key에 원본 값을 넣는 용도로 사용한다.

`RunnablePassthrough` 자체는 값을 계산하거나 딕셔너리를 만들지 않는다. 입력받은 값을 그대로 반환하는 역할만 한다.

### 기본 작성법

다음처럼 단독으로 실행하면 입력 문자열이 그대로 반환된다.

```python
passthrough = RunnablePassthrough()
passthrough.invoke('4^4은?')
# '4^4은?'
```

딕셔너리 안에 작성하면 원본 입력을 지정한 key의 값으로 보관할 수 있다.

```python
{'text': RunnablePassthrough()}
```

이 표현에 `'4^4은?'`를 입력하면 다음 딕셔너리가 만들어진다.

```python
{'text': '4^4은?'}
```

LCEL에서 딕셔너리 표현은 내부적으로 `RunnableParallel`로 변환된다. 딕셔너리가 `text` key를 만들고, `RunnablePassthrough()`가 원본 문자열을 그 key의 값으로 전달한다.

### 현재 코드에서의 처리 순서

```text
'4^4은?'
→ RunnablePassthrough가 문자열을 그대로 전달
→ {'text': '4^4은?'} 생성
→ cond_chain이 text를 읽어 math_chain 선택
```


In [15]:
from langchain_core.runnables import RunnablePassthrough

full_chain = {'text':RunnablePassthrough()} | cond_chain

pass_result = full_chain.invoke("4^2^2 결과는?")
print(pass_result)

거듭제곱은 일반적으로 **오른쪽부터** 계산합니다.

\[
4^{2^2}=4^{(2^2)}
\]

먼저,

\[
2^2=4
\]

따라서,

\[
4^4=256
\]

**최종 답: \(\boxed{256}\)**


## Message History Chain

Conditional Chain은 **한 번의 요청에서 실행할 경로**를 선택한다. 하지만 다음 요청에 앞선 질문과 답변을 자동으로 전달하지는 않는다.

Message History Chain은 이전 대화를 다시 프롬프트에 넣어 모델이 문맥을 이어서 답하도록 만든다. 모델을 다시 학습하거나 파인튜닝하는 방식과는 다르다.

### 핵심 구성 요소

- `session_id`: 어느 사용자의 대화 이력을 가져올지 구분하는 조회 키이다.
- Message History: `HumanMessage`와 `AIMessage`를 대화 순서대로 저장하는 목록이다.
- `HumanMessage`: 사용자가 입력한 질문을 나타내는 메시지 객체이다.
- `AIMessage`: Chat Model이 반환한 응답을 나타내는 메시지 객체이다.
- `MessagesPlaceholder`: 저장된 Message History가 프롬프트에 들어갈 위치이다.
- `RunnableWithMessageHistory`: Chain 실행 전에는 이력을 불러오고, 실행 후에는 새 질문과 답변을 저장하는 래퍼이다.

### 대화가 처리되는 순서

1. 요청에 포함된 `session_id`로 해당 사용자의 Message History를 찾는다.
2. 저장된 메시지를 `MessagesPlaceholder` 위치에 넣는다.
3. 이전 대화와 현재 질문을 함께 Chat Model에 전달한다.
4. 모델이 새로운 답변을 `AIMessage`로 반환한다.
5. 현재 `HumanMessage`와 새 `AIMessage`를 같은 세션 이력 끝에 추가한다.

처리 흐름은 **`session_id` → Message History 조회 → Prompt 구성 → 모델 호출 → 새 메시지 저장**이다.

### 메모리 저장 범위

`InMemoryHistory`와 `InMemoryChatMessageHistory`는 대화를 현재 Python 프로세스의 메모리에만 저장한다. 따라서 커널을 다시 시작하거나 서버가 바뀌면 이력이 사라진다.

실서비스에서는 다음 항목을 별도로 준비해야 한다.

- 데이터베이스와 같은 영속 저장소
- 사용자별 세션 접근 제어
- 대화 기록의 보존 기간과 삭제 정책

`session_id`는 대화 이력을 찾는 값이지 사용자 인증 수단은 아니다. 로그인한 사용자와 `session_id`의 소유 관계는 서버에서 별도로 확인해야 한다.

생성자 인자와 지원하는 입출력 형식은 [RunnableWithMessageHistory 공식 문서](https://reference.langchain.com/python/langchain-core/runnables/history/RunnableWithMessageHistory)에서 확인할 수 있다.

현재 설치 버전에서 `RunnableWithMessageHistory`를 만들면 deprecated 경고가 나타난다. deprecated는 즉시 실행할 수 없다는 뜻이 아니라 신규 코드에 대체 경로를 권장한다는 뜻이다. 여기서는 기존 체인에 이력을 주입하는 원리를 확인하고, 신규 챗봇의 대화 상태는 뒤의 LangGraph 단원에서 `checkpointer` 기반 persistence로 구현한다. [LangGraph Memory 공식 문서](https://docs.langchain.com/oss/python/langgraph/add-memory)에서 현재 권장 흐름을 확인할 수 있다.


### BaseChatMessageHistory를 상속해 메모리 이력 구현하기

여기서는 대화 이력 저장소가 어떻게 동작하는지 확인하기 위해 `InMemoryHistory` 클래스를 직접 구현한다.

### BaseChatMessageHistory의 역할

`BaseChatMessageHistory`는 대화 이력 저장소를 만들 때 기준이 되는 추상 기반 클래스이다. 이를 상속하는 클래스는 다음 기능을 제공해야 한다.

- `messages`: 지금까지 저장된 `HumanMessage`와 `AIMessage`의 목록이다.
- `add_messages()`: 새로운 메시지 여러 개를 기존 목록 끝에 추가한다.
- `clear()`: 해당 세션에 저장된 모든 메시지를 제거한다.

이 세 기능을 구현하면 `RunnableWithMessageHistory`가 저장소 내부 구조를 몰라도 대화 이력을 읽고 갱신할 수 있다.

### Pydantic으로 메시지 목록 관리하기

- `BaseModel`: `messages`에 선언된 자료형과 실제 값이 맞는지 검사한다.
- `Field`: Pydantic 모델 필드의 기본값이나 생성 방법을 설정한다.
- `default_factory=list`: `InMemoryHistory` 객체를 만들 때마다 새로운 빈 목록을 생성한다.

`messages = []`처럼 하나의 목록을 여러 객체가 공유하는 상황을 막기 위해 `default_factory=list`를 사용한다. 따라서 사용자마다 독립된 메시지 목록을 가질 수 있다.

### 세션별 저장 구조

`store` 딕셔너리는 `session_id`를 key로 사용하고 `InMemoryHistory` 객체를 value로 저장한다. 같은 `session_id`로 요청하면 기존 객체를 반환하고, 처음 등장한 `session_id`이면 빈 이력 객체를 생성한다.

이 저장소는 현재 Python 프로세스 안에서만 유지된다.


In [16]:

from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage
from pydantic import BaseModel, Field

# BaseChatMessageHistory: 메시지 관리 메서드 제공
# BaseModel: message 타입 검증 기능 제공
class InMemoryHistory(BaseChatMessageHistory, BaseModel):

    # 대화 이력을 저장할 목록 생성
    # BaseMessage: HumanMessage, AIMessage의 부모 클래스
    # default_factory=list: 인스턴스가 생성될 때 마다 공유하지 않는 list 생성
    messages: list[BaseMessage] = Field(default_factory=list)

    # 새 메시지를 목록 제일 뒤에 추가
    def add_messages(self, messages: list[BaseMessage]) -> None:
        self.messages.extend(messages)

    # 메시지 목록의 내용을 모두 제거
    def clear(self) -> None:
        self.messages = []



# session 별로 대화 이력을 저장하기 위한 store 생성
store: dict[str, InMemoryHistory] = {}
# {'user1': InMemoryHistory(), 'user2': InMemoryHistory()}'

# session_id가 일치하는 이력을 반환. 없으면 빈 이력 반환
def get_by_session_id(session_id:str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryHistory()

    return store[session_id]


history = get_by_session_id('user1')
print(history.messages) # user1 세션이 등록된적 없음 -> 빈 목록 반환

[]


### 대화 이력을 프롬프트에 넣고 다시 저장하기

이 단계에서는 같은 `session_id`에 저장된 이전 메시지를 현재 질문과 함께 모델에 전달하고, 새 질문과 답변을 다시 같은 세션에 저장하는 대화형 Chain을 구성한다.

#### 사용하는 구성 요소

- `MessagesPlaceholder(variable_name='history')`: 저장소에서 읽은 이전 메시지 목록이 들어갈 프롬프트 위치이다.
- `chain = prompt | llm`: 시스템 지시문, 이전 대화, 현재 질문을 모델에 전달하고 `AIMessage`를 반환한다.
- `RunnableWithMessageHistory`: Chain 실행 전에는 세션 이력을 가져오고, 실행 후에는 현재 질문과 새 답변을 같은 이력에 추가한다.

#### 처리 순서

```text
session_id로 대화 이력 조회
→ 이전 메시지를 history 위치에 삽입
→ 시스템 지시문 + 이전 대화 + 현재 질문을 모델에 전달
→ 모델이 AIMessage 반환
→ 현재 HumanMessage와 새 AIMessage를 같은 세션에 저장
```

#### AIMessage를 그대로 유지하는 이유

이 예제에서는 `StrOutputParser`를 연결하지 않으므로 모델의 출력이 문자열이 아니라 `AIMessage`로 유지된다. 이를 통해 답변 텍스트뿐 아니라 메시지 자료형과 메타데이터를 함께 확인할 수 있다.

`RunnableWithMessageHistory`는 문자열 출력도 `AIMessage`로 바꾸어 저장할 수 있다. 여기서는 **프롬프트 → Chat Model → AIMessage → Message History**의 자료형 흐름을 직접 확인하기 위해 파서를 생략한다. 화면에 필요한 답변 문자열은 다음 셀에서 `AIMessage.text`로 꺼낸다.

#### deprecated 경고의 의미

현재 설치 버전에서는 `RunnableWithMessageHistory` 객체를 만들 때 deprecated 경고가 나타난다. 기존 코드의 실행이 즉시 중단된다는 뜻은 아니며, 신규 대화 상태 관리는 LangGraph의 persistence를 사용하라는 안내이다. persistence는 이전 메시지와 같은 실행 상태를 다음 요청에서도 이어서 사용할 수 있도록 저장하는 기능이다.


In [20]:

from langchain_core.prompts import MessagesPlaceholder, ChatPromptTemplate
from langchain_core.runnables.history import RunnableWithMessageHistory

prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 {domain} 분야의 조언자이다.'),

    # 중간에 이전 대화 이력을 추가
    MessagesPlaceholder(variable_name='history'),
    ('human', '{question}'),
])

llm = ChatOpenAI(
    model=CHAT_MODEL_NAME,
    use_responses_api=True
)

# chain 결과로 llm의 응답인 AIMessage 반환
chain = prompt | llm


chain_with_history = RunnableWithMessageHistory(
    chain,  # 실행할 prompt | llm
    get_by_session_id, # session_id가 일치하는 대화 이력 찾기
    input_messages_key='question', # 사용자 입력을 전달할 key
    history_messages_key='history' # 찾은 대화 이력을 전달할 key
)

# chain_with_history.invoke(~~~)


C:\Users\playdata2\miniforge3\envs\llm_env\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


### user1의 첫 질문과 AIMessage 저장

이 호출은 `session_id='user1'`로 빈 이력을 가져와 분야와 첫 질문을 프롬프트에 넣는다.

Responses API의 AIMessage `content`는 텍스트 블록 목록일 수 있으므로, `text` 속성으로 화면에 보여 줄 문자열만 꺼낸다. 래퍼는 원래 AIMessage 객체를 user1 이력에 추가한다.


In [21]:
first_reply = chain_with_history.invoke(
    input={
        'domain': '수학',
        'question': '진희는 강아지 한 마리를 키우고 있다.'
    },
    config={'configurable': {'session_id': 'user1'}}
)

print(first_reply.text)

네, 진희가 강아지 한 마리를 키우고 있군요. 이어지는 문제를 말씀해 주세요.


### user1에 저장된 메시지 순서 확인

같은 session_id를 키로 `store['user1']`을 읽으면 첫 질문 HumanMessage와 첫 응답 AIMessage가 시간순으로 들어 있는지 확인할 수 있다. 메시지 객체 전체에는 응답 ID와 metadata도 포함되므로, 아래에서는 학습에 필요한 역할과 텍스트만 출력한다.


In [22]:
user1_history = store['user1']

for message in user1_history.messages:
    print(f'{message.type} : {message.text}')

human : 진희는 강아지 한 마리를 키우고 있다.
ai : 네, 진희가 강아지 한 마리를 키우고 있군요. 이어지는 문제를 말씀해 주세요.


### 같은 user1 세션에서 이전 문맥을 사용하기

두 번째 요청도 `session_id='user1'`을 사용하므로 RunnableWithMessageHistory는 앞 메시지를 history에 넣은 뒤 새 질문을 추가한다. 같은 세션이 문맥을 전달하는 구조를 보여 주며, 모델이 생성한 계산 결과의 정확성을 보장하지는 않는다.


In [23]:
second_reply = chain_with_history.invoke(
    input={
        'domain': '수학',
        'question': '진희는 고양이도 두 마리 키운다. 진희는 총 몇 마리의 동물을 키우는가?'
    },
    config={'configurable': {'session_id': 'user1'}}
)

print(second_reply.text)

진희는 강아지 1마리와 고양이 2마리를 키우므로, 모두 **3마리**의 동물을 키웁니다.


### 두 번째 요청 뒤 user1 이력 다시 확인

두 번째 요청 뒤에는 user1 이력에 두 번의 HumanMessage와 두 번의 AIMessage가 쌓인다. 아래에서는 네 메시지의 역할과 텍스트만 출력해 질문·응답 순서가 유지되는지 확인한다.


In [24]:
for message in user1_history.messages:
    print(f'{message.type} : {message.text}')

human : 진희는 강아지 한 마리를 키우고 있다.
ai : 네, 진희가 강아지 한 마리를 키우고 있군요. 이어지는 문제를 말씀해 주세요.
human : 진희는 고양이도 두 마리 키운다. 진희는 총 몇 마리의 동물을 키우는가?
ai : 진희는 강아지 1마리와 고양이 2마리를 키우므로, 모두 **3마리**의 동물을 키웁니다.


### 다른 session_id로 문맥을 분리하기

`session_id='user2'`는 user1과 다른 키이므로 빈 이력에서 시작한다. 같은 체인 객체를 쓰더라도 세션 ID가 다르면 이전 user1 메시지가 프롬프트에 섞이지 않아야 한다.


In [26]:
user2_reply = chain_with_history.invoke(
    input={
        'domain':'수학',
        'question':'진희는 몇 마리 동물을 키우는가?'
    },
    config={'configurable': {'session_id': 'user2'}}
)

print(user2_reply.text)

문제에 진희가 키우는 동물의 수에 대한 정보가 없어 알 수 없습니다. 관련 조건이나 지문을 알려 주세요.


## 관계 상담 세션

이번에는 직접 만든 `InMemoryHistory` 대신 LangChain Core가 같은 메시지 관리 방식을 제공하는 `InMemoryChatMessageHistory`를 사용한다.

새 딕셔너리를 대입하므로 앞의 휘발성 이력과 독립적으로 시작하며, 실서비스에서는 영속 저장소로 교체해야 한다.


### 기본 제공 저장소로 관계 상담 Chain 구성하기

앞에서 직접 구현한 대화 이력 처리 방식을 관계 상담 예제에 다시 적용한다. 이번에는 저장소만 LangChain이 제공하는 `InMemoryChatMessageHistory`로 바꾸고, 새로운 `store`에서 독립된 대화를 시작한다.

#### 주요 변수와 객체

- `store`: `session_id`별 `InMemoryChatMessageHistory` 객체를 저장하는 딕셔너리이다. 새 빈 딕셔너리를 대입하므로 앞의 대화 이력은 사용하지 않는다.
- `get_by_session_id()`: `store`에서 세션 이력을 찾아 반환하고, 처음 보는 세션이면 빈 이력을 만든다.
- `prompt`: 시스템 지시문, 이전 `history`, 현재 `question`의 순서를 정의한다.
- `llm`: 완성된 메시지 목록을 받아 `AIMessage`를 반환한다.
- `chain`: `prompt | llm`으로 연결한 관계 상담 응답 생성 경로이다.
- `chain_with_history`: `chain` 실행 전후에 세션 이력을 불러오고 저장하는 래퍼이다.

#### 처리 흐름

```text
session_id로 store의 이력 조회
→ 저장된 messages를 prompt의 history 위치에 삽입
→ system + history + question으로 모델 호출
→ AIMessage 반환
→ 현재 질문과 AIMessage를 같은 세션에 저장
```

현재 설치 버전에서는 `RunnableWithMessageHistory` 생성 시 deprecated 경고가 나타날 수 있다. 여기서는 이력 연결 원리를 확인하고, 신규 구현 방식인 LangGraph persistence는 뒤 단원에서 다룬다.


In [27]:
# 랭체인 제공 히스토리 사용
from langchain_core.chat_history import InMemoryChatMessageHistory

store: dict[str, InMemoryChatMessageHistory] = {}

def get_by_session_id(session_id:str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# prompt
prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 {domain} 분야의 조언자이다.'),
    MessagesPlaceholder(variable_name='history'),
    ('human', '{question}'),
])

llm = ChatOpenAI(
    model=CHAT_MODEL_NAME,
    use_responses_api=True
)

chain = prompt | llm

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_by_session_id,
    input_messages_key='question',
    history_messages_key='history'
)

# 메시지 이력을 저장하는 히스토리 객체만 LangChain 제공 클래스로 변경

C:\Users\playdata2\miniforge3\envs\llm_env\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


### 관계 상담 세션의 첫 인사 저장

첫 호출은 `session_id='456'`의 빈 history를 프롬프트에 넣고 인사와 이름을 포함한 현재 질문을 보낸다. 반환 AIMessage와 입력 HumanMessage는 호출 뒤 같은 세션 이력에 추가된다.


In [28]:
relation_reply = chain_with_history.invoke(
    input={
        'domain': '인간관계',
        'question': '안녕, 나는 길동이라고 해.'
    },
    config={'configurable': {'session_id': '456'}}
)

print(relation_reply.text)

안녕하세요, 길동님! 만나서 반가워요.  
인간관계에 관해 고민이 있으시면 편하게 말씀해 주세요.


### 같은 관계 상담 세션에서 앞 인사 참조하기

두 번째 호출은 첫 호출과 같은 `session_id='456'`을 사용한다. 따라서 RunnableWithMessageHistory는 첫 인사와 첫 응답을 history에 넣은 뒤 이름을 묻는 현재 질문을 모델에 전달한다.


In [29]:
relation_reply = chain_with_history.invoke(
    input={
        'domain': '인간관계',
        'question': '내 이름이 무엇이라고 했지?'
    },
    config={'configurable': {'session_id': '456'}}
)

print(relation_reply.text)

길동님이라고 했어요.


In [30]:
relation_reply = chain_with_history.invoke(
    input={
        'domain': '인간관계',
        'question': '수업 중에 자꾸 딴짓하는 친구들 어떻게 해야 될까'
    },
    config={'configurable': {'session_id': '456'}}
)

print(relation_reply.text)

길동님 수업에 방해가 될 정도라면, 감정적으로 몰아붙이기보다 단계적으로 대응해 보세요.

1. **먼저 조용히 부탁하기**  
   쉬는 시간에 “너희가 떠들면 수업을 잘 못 듣겠어. 수업 시간에는 조금만 조용히 해줄래?”처럼 친구를 비난하기보다 **내가 겪는 불편**을 중심으로 말해 보세요.

2. **수업 중에는 반응하지 않기**  
   딴짓에 같이 웃거나 반응하면 행동이 계속될 수 있어요. 가능하면 시선을 돌리고, 자리를 조금 옮기거나 수업에 집중하세요.

3. **계속되면 선생님께 알리기**  
   “친구를 혼내 달라”기보다 “수업에 집중하기 어려워서 자리 조정이나 도움을 받고 싶어요”라고 조용히 상담하는 게 좋습니다.

4. **친구 관계도 고려하기**  
   단순히 잠깐 장난치는 정도라면 너무 크게 문제 삼지 않아도 되지만, 반복적으로 방해하거나 길동님을 끌어들이고 불편하게 한다면 분명히 선을 그어야 해요.

친구들이 길동님에게 직접 말을 걸며 방해하는 건가요, 아니면 자기들끼리 떠들어서 수업을 방해하는 건가요?


In [31]:
relation_reply = chain_with_history.invoke(
    input={
        'domain': '인간관계',
        'question': '내가 선생님이야'
    },
    config={'configurable': {'session_id': '456'}}
)

print(relation_reply.text)

아, 길동님이 선생님이셨군요. 제가 잘못 이해했어요.

수업 중 딴짓이 반복될 때는 다음처럼 대응해 보세요.

- **가까이 다가가기·눈맞춤 등 비언어적 신호**로 먼저 조용히 제지하기  
- 수업 전에 “지금은 설명 듣는 시간, 활동 시간에는 자유롭게 참여”처럼 **규칙과 기대 행동을 분명히 하기**
- 딴짓하지 않고 참여하는 학생을 구체적으로 칭찬하기  
  - “지금 교과서 펴고 집중하는 모습이 좋네요.”
- 계속되면 공개적으로 망신 주기보다 수업 후 따로 불러  
  - “수업 중 무엇이 어려워서 딴짓하게 되는지”  
  - “어떻게 하면 참여할 수 있을지”를 차분히 묻기
- 자리 배치, 짧은 활동 전환, 질문·역할 부여 등으로 **수업 참여도를 높이기**
- 반복되거나 다른 학생의 학습권을 침해하면 학부모·상담교사와 협력하고, 대응 내용을 간단히 기록하기

핵심은 행동은 분명히 제한하되 학생 자체를 비난하지 않는 것입니다. “너는 왜 늘 그러니?”보다는 “지금 행동이 수업을 방해하고 있으니 멈춰야 해”처럼 말하는 편이 효과적입니다.
